In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
from scipy.stats import norm
from hmmlearn.hmm import GaussianHMM
from arch import arch_model

# Path to the clean data we just made
data_path = "../data/processed/eurusd_clean_returns_jan2026.parquet"
model_dir = "../models/"
os.makedirs(model_dir, exist_ok=True)

In [11]:
df = pd.read_parquet(data_path)
y = df['returns'].values
T = len(y)
ALPHA = 0.05
z = norm.ppf(1 - ALPHA / 2)

print(f"Loaded {T} observations. Sample Std: {np.std(y):.6f}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/eurusd_clean_returns_jan2026.parquet'

In [ ]:
""" Calculate Baselines on Real Data """
std_global = np.std(y)
mu_global = np.mean(y)

# Baseline: Always Mean
l0, u0 = mu_global - z*std_global, mu_global + z*std_global

# Baseline: Repeat Last (Momentum)
l_rep, u_rep = y[:-1] - z*std_global, y[:-1] + z*std_global
y_true = y[1:] # Align for scoring

print("Baselines ready.")

In [ ]:
""" Fit HMM to real returns """
# HMM expects (N, 1) shape
X = y.reshape(-1, 1)
hmm_model = GaussianHMM(n_components=2, covariance_type="full", n_iter=100)
hmm_model.fit(X)

# Get states and save the model
states = hmm_model.predict(X)
joblib.dump(hmm_model, os.path.join(model_dir, "hmm_eurusd_jan2026.pkl"))

print("HMM Fit Complete.")
print(f"Transition Matrix:\n{hmm_model.transmat_}")

In [ ]:
""" HMM Mixed Prediction Intervals """
probs = hmm_model.predict_proba(X) # P(S_t | observations)

# Filter parameters
mus = hmm_model.means_.flatten()
sigs = np.sqrt(hmm_model.covars_.flatten())

# Weighted prediction mean and variance
# We use probs[t-1] to predict y[t]
y_hat_hmm = np.sum(probs[:-1] * mus, axis=1)
y_var_hmm = np.sum(probs[:-1] * (sigs**2 + (mus - y_hat_hmm.reshape(-1,1))**2), axis=1)
y_std_hmm = np.sqrt(y_var_hmm)

l_hmm, u_hmm = y_hat_hmm - z*y_std_hmm, y_hat_hmm + z*y_std_hmm
print("HMM Intervals generated.")

In [ ]:
""" Fit GARCH(1,1) to real returns """
g_model = arch_model(y, mean='Zero', vol='Garch', p=1, q=1)
g_res = g_model.fit(disp='off')

# Get conditional volatility
y_std_g = g_res.conditional_volatility[1:]
l_g, u_g = 0 - z*y_std_g, 0 + z*y_std_g

joblib.dump(g_res, os.path.join(model_dir, "garch_eurusd_jan2026.pkl"))
print("GARCH Fit Complete.")

In [ ]:
""" Final Results on Real EUR/USD Data """

def score(y, l, u):
    cov = np.mean((y >= l) & (y <= u))
    wid = np.mean(u - l)
    is_v = np.mean((u - l) + (2/ALPHA)*np.maximum(l-y, 0) + (2/ALPHA)*np.maximum(y-u, 0))
    return cov, wid, is_v

results = {
    "Baseline: Global Std": score(y_true, l0, u0),
    "Baseline: Repeat Last": score(y_true, l_rep, u_rep),
    "HMM (Inferred State)": score(y_true, l_hmm, u_hmm),
    "GARCH(1,1)": score(y_true, l_g, u_g)
}

print("=" * 80)
print("REAL DATA ANALYSIS: EUR/USD JAN 2026".center(80))
print("=" * 80)
print(f"{'Model':<25} | {'Coverage':>8} | {'Avg Width':>10} | {'IS (↓)':>10}")
print("-" * 80)
for name, m in results.items():
    print(f"{name:<25} | {m[0]:>8.4f} | {m[1]:>10.4f} | {m[2]:>10.4f}")
print("=" * 80)